# Method 2 — Full pipeline & đánh giá (Kaggle T4)

Phase 5 của `docs/method2_plan.md`: chạy oracle + full pipeline, xuất predictions theo contract, đo latency 4 giai đoạn. Ngân sách ~1h GPU.

**Ba quy tắc sống còn trên Kaggle** (§8 `docs/method2_plan.md`):

1. Bật **Save & Run All (Commit)** cho job dài — session tương tác bị ngắt sau ~20 phút không tương tác, commit run chạy nền đủ 12h.
2. Checkpoint mỗi 500 step vào `/kaggle/working`, và **luôn** hỗ trợ `resume_from`.
3. Cache model HuggingFace thành Kaggle Dataset (`BAAI/bge-m3` ~2.3GB) thay vì tải lại mỗi session.


In [ ]:
# ===== Cell 1: env — PIN version =====
# Ba package này quyết định API training VÀ tên metric của
# InformationRetrievalEvaluator. Đổi bản là đổi khoá metric, hỏng cả
# load_best_model_at_end lẫn khả năng so sánh giữa các run.
!pip install -q 'transformers==5.15.1' 'sentence-transformers==6.0.0' 'peft==0.20.0' \
                accelerate jsonschema rank_bm25 datasets

# torch KHÔNG pin: Kaggle cài sẵn bản CUDA riêng, ép cài lại vừa chậm vừa
# dễ lệch CUDA runtime của image. Chỉ ghi nhận version vào manifest.
import torch

free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(0), f'{free/1024**3:.1f} / {total/1024**3:.1f} GB free')
print('bf16 supported:', torch.cuda.is_bf16_supported())
# Kỳ vọng: Tesla T4, ~15.0 GB free, bf16 = False → mọi config dùng fp16.


In [ ]:
# ===== Cell 2: mount code + data =====
# Dựng 3 dataset bằng: python scripts/method2/build_kaggle_upload.py --hf-cache
# Dataset mount ở /kaggle/input/<tên>/ với đúng cấu trúc script đã tạo.
!cp -r /kaggle/input/toolcalling-vi-src/src /kaggle/working/
!cp -r /kaggle/input/toolcalling-vi-src/configs /kaggle/working/

# toolcalling-vi-data để method2/ custom_vi/ benchmark_vi/ ở GỐC dataset,
# còn code lại tham chiếu đường dẫn tương đối `data/method2/...`.
!mkdir -p /kaggle/working/data
!cp -r /kaggle/input/toolcalling-vi-data/* /kaggle/working/data/
%cd /kaggle/working

import json, os, glob, sys
sys.path.insert(0, '/kaggle/working')

# Cache model HF thành Kaggle Dataset để không tải lại mỗi session.
# Dùng HF_HUB_CACHE chứ KHÔNG phải HF_HOME: HF_HOME cần thư mục con `hub/`,
# trong khi snapshot_download(cache_dir=...) tạo thẳng models--*/ ở gốc.
# Trỏ sai thì biến này bị bỏ qua và model vẫn tải lại từ đầu mỗi session.
HF_CACHE = '/kaggle/input/hf-cache'
if os.path.isdir(HF_CACHE):
    os.environ['HF_HUB_CACHE'] = HF_CACHE
    # Dataset của Kaggle chỉ đọc; offline tránh việc hub cố ghi file lock.
    os.environ['HF_HUB_OFFLINE'] = '1'
    print('HF cache:', sorted(os.listdir(HF_CACHE))[:5])
else:
    print('Không có dataset hf-cache — model sẽ tải từ Hub (chậm hơn, có rate limit)')

manifest = json.load(open('data/method2/manifest.json', encoding='utf-8'))
print('snapshot commit:', manifest.get('git_commit'))


## Hai chế độ bắt buộc (§6.3 experimental_plan)

| Chế độ | Input Cross-Encoder | Trả lời câu hỏi |
|---|---|---|
| `oracle` | Tool gold | Extraction tốt đến đâu, độc lập retrieval |
| `pipeline` | Bi-Encoder top-k + abstention | Hiệu năng hệ thống thật |

`ArgA_oracle − ArgA_pipeline` = phần lỗi do retrieval. Con số này phải xuất
hiện tường minh trong báo cáo.


In [ ]:
import subprocess

# subprocess thay vì `!` trong vòng lặp: exit code hiện ra rõ ràng nên một
# lần chạy hỏng không bị trôi qua trong Save & Run All.
for gold, tag in [('data/custom_vi/v1/test_seen.jsonl', 'custom_seen'),
                  ('data/custom_vi/v1/test_unseen.jsonl', 'custom_unseen'),
                  ('data/benchmark_vi/test.jsonl', 'benchmark')]:
    for mode in ['pipeline', 'oracle']:
        print(f'=== {tag} / {mode} ===', flush=True)
        subprocess.run(
            ['python', '-m', 'src.models.pipeline.method2',
             '--config', 'configs/method2/pipeline.yaml',
             '--gold', gold, '--mode', mode,
             '--output-dir', f'results/method2/predictions/{tag}'],
            check=True,
        )


## Chạy evaluator chung

Cùng normalization, cùng rule so khớp với ba method còn lại — xem
`docs/evaluation.md`.


In [ ]:
!python -m src.evaluation.cli evaluate \
    --gold data/custom_vi/v1/test_seen.jsonl \
    --predictions results/method2/predictions/custom_seen/predictions.jsonl \
    --oracle-predictions results/method2/predictions/custom_seen/oracle_predictions.jsonl \
    --slice metadata.tool_split \
    --output-dir results/evaluation/method_2_seen


## Latency tách 4 giai đoạn — `t_index_build` ghi riêng, không cộng vào


In [ ]:
latency = json.load(open('results/method2/predictions/custom_seen/latency_pipeline.json', encoding='utf-8'))
for stage in ['t_query_embed', 't_retrieve', 't_cross_encode', 't_validate', 'total']:
    print(f"{stage:16s} p50={latency[stage]['p50_ms']:8.2f} ms  p95={latency[stage]['p95_ms']:8.2f} ms")

index_meta = json.load(open('data/method2/index/index_meta.json', encoding='utf-8'))
print('\nt_index_build (một lần, KHÔNG cộng vào latency/query):', index_meta['t_index_build_sec'], 's')


In [ ]:
# ===== Lưu artifact =====
# Kaggle chỉ giữ /kaggle/working (20GB). Nén để tải về hoặc làm Dataset mới.
!tar czf /kaggle/working/eval_run.tar.gz -C /kaggle/working/artifacts/method2 .
!du -h /kaggle/working/eval_run.tar.gz
